In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [9]:
!ls /content/drive/MyDrive/OIMHS_masks_test/OIMHS_masks_test

CNV  DRUSEN  NORMAL


In [13]:
import torch
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

num_classes = 3  # CNV, DRUSEN, NORMAL

model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

ckpt = torch.load("/content/oct_oimhs_mobilnetv2.pth", map_location=device)

# dacă e salvat ca dict cu "state_dict"
if "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

# scoate prefix "module." dacă a fost antrenat pe DataParallel
new_state = {}
for k,v in state_dict.items():
    nk = k.replace("module.", "")
    new_state[nk] = v

model.load_state_dict(new_state, strict=False)
model.to(device)
model.eval()
print("Model loaded OK")


Using: cuda
Model loaded OK


In [14]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

test_dir = "/content/drive/MyDrive/OIMHS_masks_test/OIMHS_masks_test"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

test_data = datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = test_data.classes
print("Classes:", class_names)


Classes: ['CNV', 'DRUSEN', 'NORMAL']


In [15]:
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)
        print(f"Corecte: {correct} din {total}")


        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = correct / total * 100
print(f"Acuratețe: {accuracy:.2f}%")


Corecte: 30 din 32
Corecte: 61 din 64
Corecte: 93 din 96
Corecte: 125 din 128
Corecte: 156 din 160
Corecte: 188 din 192
Corecte: 219 din 224
Corecte: 238 din 256
Corecte: 238 din 288
Corecte: 239 din 320
Corecte: 239 din 352
Corecte: 239 din 384
Corecte: 239 din 416
Corecte: 239 din 448
Corecte: 239 din 480
Corecte: 239 din 512
Corecte: 239 din 544
Corecte: 239 din 576
Corecte: 239 din 608
Corecte: 239 din 640
Corecte: 239 din 672
Corecte: 239 din 704
Corecte: 239 din 726
Acuratețe: 32.92%


In [17]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

class_names = ["CNV", "DRUSEN", "NORMAL"]

# === Classification Report ===
print("\nClassification Report :\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

# === Acuratețe per clasă ===
cm = confusion_matrix(all_labels, all_preds)
class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("\n=== Acuratețe per clasă ===")
for cls, acc in zip(class_names, class_accuracy):
    print(f"{cls}: {acc:.4f}")



Classification Report :

              precision    recall  f1-score   support

         CNV       0.33      0.98      0.49       242
      DRUSEN       0.25      0.01      0.02       242
      NORMAL       0.00      0.00      0.00       242

    accuracy                           0.33       726
   macro avg       0.19      0.33      0.17       726
weighted avg       0.19      0.33      0.17       726


=== Acuratețe per clasă ===
CNV: 0.9793
DRUSEN: 0.0083
NORMAL: 0.0000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [18]:
import torch
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

num_classes = 3  # CNV, DRUSEN, NORMAL

model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

ckpt = torch.load("/content/oct_oimhs_resnet18.pth", map_location=device)

# dacă e salvat ca dict cu "state_dict"
if "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

# scoate prefix "module." dacă a fost antrenat pe DataParallel
new_state = {}
for k,v in state_dict.items():
    nk = k.replace("module.", "")
    new_state[nk] = v

model.load_state_dict(new_state, strict=False)
model.to(device)
model.eval()
print("Model loaded OK")


Using: cuda
Model loaded OK


In [19]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

test_dir = "/content/drive/MyDrive/OIMHS_masks_test/OIMHS_masks_test"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

test_data = datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = test_data.classes
print("Classes:", class_names)


Classes: ['CNV', 'DRUSEN', 'NORMAL']


In [20]:
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)
        print(f"Corecte: {correct} din {total}")


        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = correct / total * 100
print(f"Acuratețe: {accuracy:.2f}%")


Corecte: 32 din 32
Corecte: 64 din 64
Corecte: 95 din 96
Corecte: 126 din 128
Corecte: 158 din 160
Corecte: 190 din 192
Corecte: 220 din 224
Corecte: 251 din 256
Corecte: 280 din 288
Corecte: 307 din 320
Corecte: 337 din 352
Corecte: 368 din 384
Corecte: 397 din 416
Corecte: 425 din 448
Corecte: 454 din 480
Corecte: 484 din 512
Corecte: 515 din 544
Corecte: 543 din 576
Corecte: 573 din 608
Corecte: 599 din 640
Corecte: 628 din 672
Corecte: 654 din 704
Corecte: 674 din 726
Acuratețe: 92.84%


In [21]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

class_names = ["CNV", "DRUSEN", "NORMAL"]

# === Classification Report ===
print("\nClassification Report :\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

# === Acuratețe per clasă ===
cm = confusion_matrix(all_labels, all_preds)
class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("\n=== Acuratețe per clasă ===")
for cls, acc in zip(class_names, class_accuracy):
    print(f"{cls}: {acc:.4f}")



Classification Report :

              precision    recall  f1-score   support

         CNV       0.92      0.98      0.95       242
      DRUSEN       0.88      0.91      0.89       242
      NORMAL       1.00      0.89      0.94       242

    accuracy                           0.93       726
   macro avg       0.93      0.93      0.93       726
weighted avg       0.93      0.93      0.93       726


=== Acuratețe per clasă ===
CNV: 0.9835
DRUSEN: 0.9091
NORMAL: 0.8926
